# 02 · Features

Construye las tablas que usan el backtest (03) y las predicciones semanales (04). El código vive en el paquete `src/fantasy_ml/` (`scoring.py`, `features.py`), para que todos los notebooks usen exactamente las mismas funciones; este notebook documenta y valida.

**Parte 1 (este notebook, primera sección): puntos de K y D/ST.** nflverse trae `fantasy_points_ppr` para QB/RB/WR/TE, pero no para kickers ni defensas. Los calculo con `config/scoring.yaml` y **valido el cálculo contra los puntos reales de ESPN** en 2026.

**Parte 2: features** de forma reciente, uso, contexto del partido y rival para QB/RB/WR/TE, K y D/ST, calculadas solo con partidos anteriores y con una prueba automática de fuga de información.

> Solo temporada regular. Las credenciales se leen de `.env` y nunca se imprimen.

## 1. Setup

In [ ]:
import polars as pl
from polars.testing import assert_frame_equal

from fantasy_ml import data, scoring, espn, features as F
from fantasy_ml.data import KEYS, DATA_PROC

SCORING = data.load_config("scoring")
VALIDATION = data.load_config("validation")
REFRESH = False  # True para volver a descargar (2026 está en curso: refrescar cada semana)

pl.Config.set_tbl_rows(20)

### Esquema de validación (`config/validation.yaml`)

Walk-forward **semanal**: para cada semana evaluada se entrena con todos los partidos anteriores y se predice esa semana (notebook 03). Los hiperparámetros se eligen solo con 2024 y quedan congelados para 2025 y 2026. La liga es nueva en 2026, así que solo 2026 se compara también contra las proyecciones de ESPN.

In [ ]:
pl.DataFrame([{"temporada": e["season"], "rol": e["role"], "compara contra": ", ".join(e["compare_against"])}
              for e in VALIDATION["evaluation"]])

## 2. Datos

In [ ]:
src = data.load_sources(refresh=REFRESH)  # caché en data/raw/
player_stats, team_stats, schedules = src["player_stats"], src["team_stats"], src["schedules"]

for name, df in src.items():
    print(f"{name:13s} {df.height:>7,} filas")

## 3. Puntos de kicker

| regla | columnas de nflverse |
|---|---|
| FG 0–39 (3) | `fg_made_0_19` + `fg_made_20_29` + `fg_made_30_39` |
| FG 40–49 (4), 50–59 (5), 60+ (6) | `fg_made_40_49`, `fg_made_50_59`, `fg_made_60_` |
| FG fallado (−1) | `fg_missed` **+ `fg_blocked`** |
| PAT (1) | `pat_made` |

ESPN cuenta un FG bloqueado como fallado; en nflverse `fg_missed` no incluye los bloqueados. Lo descubrí en la validación contra ESPN (sección 5).

In [ ]:
points_k = scoring.k_points(player_stats, SCORING)  # src/fantasy_ml/scoring.py

print(f"{points_k.height:,} partidos de kicker")
points_k.group_by("season").agg(pl.len().alias("partidos"), pl.col("fantasy_points").mean().round(2).alias("media"),
                                 pl.col("fantasy_points").max().alias("max")).sort("season")

## 4. Puntos de D/ST

**Eventos** (de `team_stats` del propio equipo): sacks, INTs, fumbles recuperados, safeties, bloqueos y TDs. En los TDs, nflverse separa `def_tds`, `fumble_recovery_tds` y `special_teams_tds`, así que sumo las tres columnas.

**Puntos y yardas permitidas:** las calculo con la definición de ESPN, que obtuve comparando con los valores reales de ESPN en la sección 5:
- **Puntos permitidos** = marcador del rival − 6 × (TDs de su defensa + TDs tras recuperar un fumble) − 2 × sus safeties. ESPN no le cuenta a tu D/ST los puntos que anota la defensa rival contra tu ofensiva.
- **Yardas permitidas** = yardas de pase del rival − yardas perdidas en sacks + yardas de carrera. Ojo: en nflverse `sack_yards_lost` viene con signo negativo, por eso uso el valor absoluto.

In [ ]:
points_dst = scoring.dst_points(team_stats, schedules, SCORING)  # src/fantasy_ml/scoring.py

n_null = points_dst["fantasy_points"].null_count()
print(f"{points_dst.height:,} partidos de D/ST · sin datos de team_stats: {n_null}")
assert n_null == 0, "Hay partidos sin team_stats: revisar joins"
points_dst.group_by("season").agg(pl.len().alias("partidos"), pl.col("fantasy_points").mean().round(2).alias("media"),
                                   pl.col("fantasy_points").min().alias("min"), pl.col("fantasy_points").max().alias("max")).sort("season")

## 5. Validación contra los puntos reales de ESPN (2026)

Para cada K y cada D/ST de la NFL descargo con `league.player_info` los puntos que ESPN les asignó en cada semana ya jugada de 2026 y los comparo con mi cálculo. Guardo también el detalle de ESPN (puntos y yardas permitidas, sacks) para poder explicar cada diferencia.

Así descubrí las tres reglas de las secciones 3 y 4 (FG bloqueado = fallado; ESPN excluye los TDs y safeties de la defensa rival de los puntos permitidos; signo de `sack_yards_lost`).

In [ ]:
league = espn.connect(2026)

# IDs de ESPN: los D/ST usan -16000 - proTeamId; los kickers se mapean con ff_playerids
dst_ids = F.dst_espn_ids()
ff_ids = F.gsis_to_espn(src["playerids"])

k26 = points_k.filter(pl.col("season") == 2026).join(ff_ids, on="player_id", how="left")
WEEKS = sorted(points_dst.filter(pl.col("season") == 2026)["week"].unique().to_list())
print(f"Semanas 2026 jugadas: {WEEKS} · K sin espn_id: {k26['espn_id'].null_count()}")

In [ ]:
actuals_path = data.DATA_RAW / "espn_actuals_k_dst_2026.parquet"
need_fetch = REFRESH or not actuals_path.exists() or \
             sorted(pl.read_parquet(actuals_path)["week"].unique().to_list()) != WEEKS

if need_fetch:
    ids = k26["espn_id"].drop_nulls().unique().to_list() + dst_ids["espn_id"].to_list()
    rows = []
    for i in range(0, len(ids), 25):
        res = league.player_info(playerId=ids[i:i + 25])
        for p in res if isinstance(res, list) else [res]:
            for wk in WEEKS:
                wk_stats = p.stats.get(wk, {})
                if "points" not in wk_stats:
                    continue
                raw = wk_stats.get("breakdown", {})
                rows.append({"espn_id": p.playerId, "name": p.name, "pos": p.position, "week": wk,
                             "espn_points": float(wk_stats["points"]),
                             "espn_pa": raw.get("defensivePointsAllowed"), "espn_ya": raw.get("defensiveYardsAllowed"),
                             "espn_sacks": raw.get("defensiveSacks")})
    actuals = pl.DataFrame(rows, schema_overrides={"espn_pa": pl.Float64, "espn_ya": pl.Float64, "espn_sacks": pl.Float64})
    actuals.write_parquet(actuals_path)
else:
    actuals = pl.read_parquet(actuals_path)
actuals.group_by("pos").agg(pl.len().alias("partidos"), pl.col("espn_id").n_unique().alias("jugadores"))

In [ ]:
TOL = 1e-6
MIN_MATCH = 0.95  # por debajo de esto la fórmula está mal, no son simples correcciones de datos

val_k = (actuals.filter(pl.col("pos") == "K")
         .join(k26.select("espn_id", "week", "player_display_name", "fg_missed", "fg_blocked", "fantasy_points"),
               on=["espn_id", "week"], how="left")
         .with_columns(diff=pl.col("espn_points") - pl.col("fantasy_points")))
val_dst = (actuals.filter(pl.col("pos") == "D/ST").join(dst_ids, on="espn_id")
           .join(points_dst.filter(pl.col("season") == 2026), on=["team", "week"], how="left")
           .with_columns(diff=pl.col("espn_points") - pl.col("fantasy_points")))

summary = []
for name, v in [("K", val_k), ("D/ST", val_dst)]:
    ok = (v["diff"].abs() < TOL).sum()
    summary.append({"posición": name, "partidos": v.height, "coinciden": ok, "pct": round(ok / v.height * 100, 1),
                    "sin cálculo": v["fantasy_points"].null_count()})
summary.append({"posición": "D/ST: puntos permitidos", "partidos": val_dst.height,
                "coinciden": (val_dst["points_allowed"] == val_dst["espn_pa"]).sum(), "pct": None, "sin cálculo": None})
summary.append({"posición": "D/ST: yardas permitidas", "partidos": val_dst.height,
                "coinciden": (val_dst["yards_allowed"] == val_dst["espn_ya"]).sum(), "pct": None, "sin cálculo": None})
summary = pl.DataFrame(summary)

for row in summary.filter(pl.col("pct").is_not_null()).iter_rows(named=True):
    if row["pct"] / 100 < MIN_MATCH:
        raise ValueError(f"{row['posición']}: solo {row['pct']}% coincide con ESPN: revisar la fórmula")
summary

Diferencias restantes y su causa probable:

In [ ]:
(pl.concat([
    val_k.filter(pl.col("diff").abs() > TOL)
         .select(pl.col("name"), "week", "espn_points", "fantasy_points", "diff",
                 detalle=pl.format("fg_missed={} fg_blocked={}", "fg_missed", "fg_blocked")),
    val_dst.filter(pl.col("diff").abs() > TOL)
           .select(pl.col("name"), "week", "espn_points", "fantasy_points", "diff",
                   detalle=pl.format("sacks nflverse={} / ESPN={} · PA {} / {} · YA {} / {}",
                                     "def_sacks", "espn_sacks", "points_allowed", "espn_pa", "yards_allowed", "espn_ya")),
], how="vertical_relaxed"))

Si solo queda alguna diferencia puntual en una estadística (por ejemplo, un sack de más en ESPN) y los puntos y yardas permitidas cuadran, es una discrepancia entre las fuentes de datos o una corrección de estadísticas posterior, no un error de la fórmula.

## 6. Guardar

In [ ]:
points_dst = points_dst.join(dst_ids, on="team", how="left")  # espn_id para cruzar con rosters de ESPN
points_k = points_k.join(ff_ids, on="player_id", how="left")

DATA_PROC.mkdir(parents=True, exist_ok=True)
points_k.write_parquet(DATA_PROC / "points_k.parquet")
points_dst.write_parquet(DATA_PROC / "points_dst.parquet")
for p in ["points_k.parquet", "points_dst.parquet"]:
    print(f"✓ data/processed/{p}")

## 7. Tabla base de QB/RB/WR/TE

Una fila por jugador y partido de temporada regular.

- **Partidos sin estadísticas:** `player_stats` solo tiene filas de jugadores que registraron alguna estadística. Un TE que jugó 30 snaps bloqueando no aparece, y eso inflaría los promedios. Por eso uno la tabla con `snap_counts` y agrego esos partidos con 0 en todas las estadísticas.
- **Puntos esperados (`xfp`):** vienen de `ff_opportunity` y son los puntos que suele producir ese volumen de uso (targets, carries, distancia al end zone), sin la suerte de los TDs.

In [ ]:
base = F.offense_base(player_stats, src["snap_counts"], src["opportunity"], src["playerids"])

print(f"{base.height:,} partidos · agregados desde snap_counts con 0 puntos: {base['from_snaps_only'].sum():,} · "
      f"sin xfp: {base['xfp'].null_count():,} · sin % de snaps: {base['offense_pct'].null_count():,}")
base.group_by("position").agg(pl.len().alias("partidos"), pl.col("fantasy_points_ppr").mean().round(2).alias("ppr_media"),
                              pl.col("from_snaps_only").mean().round(3).alias("pct_solo_snaps")).sort("position")

## 8. Contexto del partido (`schedules`)

Una fila por equipo y partido, **incluidos los que aún no se juegan**, para poder predecir la semana siguiente en el notebook 05.

- **Líneas de apuestas:** `spread_line` es positivo cuando el local es favorito (su correlación con el resultado es positiva). Con él y el total calculo los puntos que se espera que anote cada equipo: `implied_team = (total + spread) / 2`, desde la perspectiva de cada equipo.
- **Estadio techado (`indoor`):** domo o techo cerrado. En esos partidos pongo viento = 0 y temperatura = 70 °F, porque nflverse los deja vacíos.
- **Clima faltante:** nflverse no trae el clima de unos 50 partidos jugados al aire libre (sobre todo de 2023). Los dejo como nulos; los modelos de árboles los manejan bien.
- ⚠ Las líneas son **de cierre** (ver limitaciones).

In [ ]:
team_games = F.team_games(schedules)
context = F.game_context(team_games)

played = context.join(team_games.filter(pl.col("team_score").is_not_null()).select(*KEYS, "team"), on=[*KEYS, "team"])
print(f"{context.height:,} filas equipo-partido ({played.height:,} ya jugadas)")
print(f"  jugadas sin línea de apuestas: {played['spread'].null_count()} · jugadas sin clima: {played['wind'].null_count()}")
print(f"  futuras sin línea todavía: {context['spread'].null_count() - played['spread'].null_count()}")

## 9. Features sin fuga de información

Implementación: `src/fantasy_ml/features.py`. Todas las features de forma reciente se calculan con `shift(1)` dentro de cada jugador o equipo: la fila de la semana *w* solo ve partidos anteriores.

| sufijo | significado |
|---|---|
| `_l3`, `_l5` | media de los últimos 3 / 5 partidos jugados (cruza temporadas) |
| `_std` | media de la temporada hasta la semana anterior (*season to date*) |
| `_prev` | media de toda la temporada anterior |
| `games_season`, `games_career` | partidos previos en la temporada / en total (0 = debut) |
| `weeks_since_last` | semanas desde su último partido en la temporada (>1 = bye o ausencia) |

Las medias `_l3` y `_l5` cruzan temporadas a propósito: en la semana 1 usan el final de la temporada anterior. `games_season` le dice al modelo cuánto de eso es del año actual.

**Rival:** para cada defensa y posición sumo los puntos PPR que permitió por partido y calculo su media de los últimos 5 partidos y de la temporada (por ejemplo, cuántos puntos suele permitir a los WR).

In [ ]:
# Implementación en src/fantasy_ml/features.py: add_rolling (medias con shift(1)), add_counts y build_*
import inspect
print(inspect.getsource(F.add_rolling))

### QB/RB/WR/TE

In [ ]:
features_offense = F.build_offense(base, context)
OFF_FEATS = F.feature_cols(features_offense)
print(f"{features_offense.height:,} filas · {len(OFF_FEATS)} features")

### K y D/ST

In [ ]:
team_offense = F.team_offense(team_games, team_stats)
features_k = F.build_k(points_k, context)
features_dst = F.build_dst(points_dst, team_offense, context)
K_FEATS, DST_FEATS = F.feature_cols(features_k), F.feature_cols(features_dst)
print(f"K:    {features_k.height:,} filas · {len(K_FEATS)} features")
print(f"D/ST: {features_dst.height:,} filas · {len(DST_FEATS)} features")

## 10. Pruebas

### Fuga de información
Multiplico por 100 **todas** las estadísticas de una semana (2025, semana 10) y reconstruyo las features. Si no hay fuga:
- las features de esa semana y de las anteriores no cambian (no pueden ver el dato alterado);
- las de semanas posteriores **sí** cambian, lo que confirma que la prueba detectaría una fuga si la hubiera.

In [ ]:
LEAK_SEASON, LEAK_WEEK = 2025, 10
is_leak_week = (pl.col("season") == LEAK_SEASON) & (pl.col("week") == LEAK_WEEK)
up_to = (pl.col("season") < LEAK_SEASON) | ((pl.col("season") == LEAK_SEASON) & (pl.col("week") <= LEAK_WEEK))

def corrupt(df, cols):
    return df.with_columns([pl.when(is_leak_week).then(pl.col(c) * 100 + 7).otherwise(pl.col(c)).alias(c) for c in cols])

def leakage_test(name, original, rebuilt, keys, feats):
    a, b = original.sort(keys), rebuilt.sort(keys)
    assert_frame_equal(a.filter(up_to).select(feats), b.filter(up_to).select(feats))
    changed = not a.filter(~up_to).select(feats).equals(b.filter(~up_to).select(feats))
    assert changed, f"{name}: la alteración no afectó semanas posteriores; la prueba no es sensible"
    print(f"✓ {name}: sin fuga (features hasta {LEAK_SEASON} sem {LEAK_WEEK} idénticas; posteriores cambian)")

num = lambda df, exclude: [c for c, t in df.schema.items() if t.is_numeric() and c not in exclude]

leakage_test("QB/RB/WR/TE", features_offense,
             F.build_offense(corrupt(base, num(base, KEYS)), context), ["player_id", *KEYS], OFF_FEATS)
leakage_test("K", features_k,
             F.build_k(corrupt(points_k, num(points_k, [*KEYS, "espn_id"])), context), ["player_id", *KEYS], K_FEATS)
leakage_test("D/ST", features_dst,
             F.build_dst(corrupt(points_dst, num(points_dst, [*KEYS, "espn_id"])),
                         corrupt(team_offense, num(team_offense, KEYS)), context), ["team", *KEYS], DST_FEATS)

### Duplicados y nulos

In [ ]:
for name, df, keys in [("QB/RB/WR/TE", features_offense, ["player_id", *KEYS]),
                       ("K", features_k, ["player_id", *KEYS]), ("D/ST", features_dst, ["team", *KEYS])]:
    assert df.select(keys).is_duplicated().sum() == 0, f"{name}: filas duplicadas"
    assert df["y"].null_count() == 0, f"{name}: objetivo nulo"
print("✓ Sin duplicados ni objetivos nulos")

# Nulos esperados: primeras apariciones (sin partidos previos) y partidos sin línea/clima
def null_report(df, feats):
    return (df.select(pl.col(feats).null_count()).transpose(include_header=True, header_name="feature", column_names=["nulos"])
              .with_columns((pl.col("nulos") / df.height * 100).round(1).alias("pct")).sort("nulos", descending=True))

null_report(features_offense, OFF_FEATS).head(10)

Los `_prev` son los que más nulos tienen, lo cual es esperable: son todas las filas de 2023 (no hay 2022 en los datos) más los novatos. Los `_std` son nulos en el primer partido de cada temporada.

### ¿Las features tienen sentido?
Correlación de algunas features con el objetivo, por posición. Es solo una comprobación de sentido común, no una selección de features.

In [ ]:
CHECK = ["fantasy_points_ppr_l5", "xfp_l5", "fantasy_points_ppr_std", "offense_pct_l3", "implied_team", "opp_ppr_allowed_l5"]
(features_offense.filter(pl.col("games_career") >= 3)
 .group_by("position")
 .agg([pl.corr(c, "y").round(3).alias(c) for c in CHECK])
 .sort("position"))

## 11. Guardar

In [ ]:
for name, df in [("features_offense", features_offense), ("features_k", features_k), ("features_dst", features_dst)]:
    df.write_parquet(DATA_PROC / f"{name}.parquet")
    print(f"✓ data/processed/{name}.parquet · {df.height:,} filas × {df.width} columnas")

Cada tabla tiene **identificadores**, el **objetivo `y`** (puntos del partido) y **features**, pero no las estadísticas del propio partido. Así el notebook 04 no puede usarlas por error. Las features son todas las columnas que no son identificadores ni `y`.

## Supuestos y limitaciones

- **Las líneas de apuestas son de cierre.** `schedules` de nflverse trae el spread y el total *de cierre* (justo antes del partido). Si la alineación se decide días antes, solo estarán las líneas de apertura o las del momento, así que el backtest es algo optimista respecto al uso real.
- **El modelo supone que el jugador juega.** El objetivo solo existe para partidos jugados: el modelo predice los puntos *si juega*, no la probabilidad de que juegue. Lesiones, inactivos y descansos se manejan fuera del modelo (con el estado de lesión de ESPN al decidir la alineación).
- **Validación de K y D/ST limitada a 2026.** La fórmula se verificó contra ESPN solo en las semanas jugadas de 2026, porque la liga es nueva. Para 2023–2025 se asume que las reglas de ESPN no cambiaron.
- **Casos raros de D/ST sin verificar:** si ESPN resta de los puntos permitidos los TDs de retorno (kickoff o punt) del rival, y cómo registra nflverse los retornos de 2 pts (`def_2pt_made`). No aparecieron en la muestra de validación. La safety de 1 punto no se modela.
- **Correcciones de estadísticas.** ESPN y nflverse pueden diferir en algún dato puntual (por ejemplo, sacks compartidos o corregidos después del partido).